# YOLOv8 Traffic Object Detection Project

This notebook shows a simple and beginner-friendly workflow for a YOLOv8 traffic object detection project. It checks the dataset, trains YOLOv8n, loads the best model, runs predictions, and shows the main output images.

## 1. Install and Import Required Libraries

In [ ]:
# Run this cell once if the required libraries are not installed.
%pip install ultralytics pyyaml pillow matplotlib -q

# Import the libraries used in this notebook.
from pathlib import Path
import yaml
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO

## 2. Set Dataset Path and YAML Path

In [ ]:
# Set the main paths using Windows-friendly raw strings.
dataset_path = Path(r"C:\Users\USER\OneDrive\CUD\Semesters\SP 2025-2026\AI\The Code\Traffic Dataset.yolov8")
yaml_path = dataset_path / "data.yaml"
project_root = dataset_path.parent
runs_path = dataset_path / "runs"
detect_root = project_root / "runs" / "detect"

print("Dataset path:", dataset_path)
print("YAML path:", yaml_path)
print("Runs path:", runs_path)
print("Detect root:", detect_root)

## 3. Check Dataset Contents

In [ ]:
# Check that the most important folders and files are available.
paths_to_check = {
    "dataset folder": dataset_path,
    "data.yaml": yaml_path,
    "train images": dataset_path / "train" / "images",
    "train labels": dataset_path / "train" / "labels",
    "valid images": dataset_path / "valid" / "images",
    "valid labels": dataset_path / "valid" / "labels",
    "runs folder": runs_path,
}

for name, path in paths_to_check.items():
    status = "Found" if path.exists() else "Missing"
    print(f"{name}: {status} -> {path}")

## 4. Print Train and Validation Image and Label Counts

In [ ]:
# Count image and label files in the dataset folders.
image_exts = {".jpg", ".jpeg", ".png", ".bmp"}
label_exts = {".txt"}

def count_files(folder_path, allowed_exts):
    if not folder_path.exists():
        return 0
    return sum(1 for file in folder_path.iterdir() if file.is_file() and file.suffix.lower() in allowed_exts)

train_images_path = dataset_path / "train" / "images"
train_labels_path = dataset_path / "train" / "labels"
valid_images_path = dataset_path / "valid" / "images"
valid_labels_path = dataset_path / "valid" / "labels"

train_image_count = count_files(train_images_path, image_exts)
train_label_count = count_files(train_labels_path, label_exts)
valid_image_count = count_files(valid_images_path, image_exts)
valid_label_count = count_files(valid_labels_path, label_exts)

print(f"Train images: {train_image_count}")
print(f"Train labels: {train_label_count}")
print(f"Validation images: {valid_image_count}")
print(f"Validation labels: {valid_label_count}")

## 5. Show the Cleaned data.yaml Content

In [ ]:
# Load and print the YAML content in a clean way.
if yaml_path.exists():
    with open(yaml_path, "r", encoding="utf-8") as file:
        data_config = yaml.safe_load(file)
    print(yaml.dump(data_config, sort_keys=False, allow_unicode=False))
else:
    print("data.yaml was not found.")

## 6. Train YOLOv8n for 30 Epochs

In [ ]:
# Train YOLOv8n for 30 epochs and save the run as traffic_yolov8n.
# If the folder already exists, exist_ok=True helps the cell run without crashing.
train_results_30 = None

try:
    model_30 = YOLO("yolov8n.pt")
    train_results_30 = model_30.train(
        data=str(yaml_path),
        epochs=30,
        imgsz=640,
        project=str(runs_path),
        name="traffic_yolov8n",
        exist_ok=True,
    )
    print("30-epoch training finished.")
    print("Saved to:", runs_path / "traffic_yolov8n")
except Exception as error:
    print("Training could not be completed:")
    print(error)

## 7. Train YOLOv8n for 50 Epochs

In [ ]:
# Train YOLOv8n for 50 epochs and save the run as traffic_yolov8n_50epochs.
train_results_50 = None

try:
    model_50 = YOLO("yolov8n.pt")
    train_results_50 = model_50.train(
        data=str(yaml_path),
        epochs=50,
        imgsz=640,
        project=str(runs_path),
        name="traffic_yolov8n_50epochs",
        exist_ok=True,
    )
    print("50-epoch training finished.")
    print("Saved to:", runs_path / "traffic_yolov8n_50epochs")
except Exception as error:
    print("Training could not be completed:")
    print(error)

## 8. Load the Best Model from the 50-Epoch Run

In [ ]:
# Load best.pt from the 50-epoch run.
best_model_path = runs_path / "traffic_yolov8n_50epochs" / "weights" / "best.pt"
best_model = None

if best_model_path.exists():
    try:
        best_model = YOLO(str(best_model_path))
        print("Best model loaded successfully.")
        print("Model path:", best_model_path)
    except Exception as error:
        print("The model file exists, but loading failed.")
        print(error)
else:
    print("best.pt was not found.")
    print("Expected path:", best_model_path)

## 9. Run Prediction on One Validation Image

In [ ]:
# Use one validation image and show the prediction inside the notebook.
image_exts = {".jpg", ".jpeg", ".png", ".bmp"}

if valid_images_path.exists():
    valid_image_files = sorted(
        [file for file in valid_images_path.iterdir() if file.is_file() and file.suffix.lower() in image_exts]
    )
else:
    valid_image_files = []

if best_model is None:
    print("Best model is not loaded yet.")
elif not valid_image_files:
    print("No validation images were found.")
else:
    sample_image = valid_image_files[0]
    print("Using image:", sample_image.name)
    sample_results = best_model.predict(source=str(sample_image), save=False, verbose=False)
    sample_plot = sample_results[0].plot()

    plt.figure(figsize=(10, 8))
    plt.imshow(sample_plot[:, :, ::-1])
    plt.title(f"Prediction for {sample_image.name}")
    plt.axis("off")
    plt.show()

## 10. Run Prediction on the First 5 Validation Images with conf=0.27

In [ ]:
# Predict the first 5 validation images and save them to the detect folder.
# Using exist_ok=False allows YOLO to create a new predict folder like predict7 if needed.
latest_predict_folder = None

if best_model is None:
    print("Best model is not loaded yet.")
elif not valid_image_files:
    print("No validation images were found.")
else:
    predict_source = [str(file) for file in valid_image_files[:5]]
    detect_root.mkdir(parents=True, exist_ok=True)

    try:
        prediction_results = best_model.predict(
            source=predict_source,
            conf=0.27,
            save=True,
            project=str(detect_root),
            name="predict",
            exist_ok=False,
            verbose=False,
        )
        print("Prediction for the first 5 validation images is complete.")
    except Exception as error:
        print("Prediction could not be completed:")
        print(error)

predict_folders = sorted(
    [folder for folder in detect_root.glob("predict*") if folder.is_dir()],
    key=lambda folder: folder.stat().st_mtime,
    reverse=True,
)

if predict_folders:
    latest_predict_folder = predict_folders[0]
    print("Latest predict folder:", latest_predict_folder)
else:
    print("No predict folders were found.")

## 11. Display Training Result Images

In [ ]:
# Show the main evaluation output images from the 50-epoch run.
results_folder = runs_path / "traffic_yolov8n_50epochs"
evaluation_files = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png",
]

for file_name in evaluation_files:
    file_path = results_folder / file_name
    print(f"\n{file_name}")

    if file_path.exists():
        plt.figure(figsize=(10, 6))
        plt.imshow(Image.open(file_path))
        plt.title(file_name)
        plt.axis("off")
        plt.show()
    else:
        print("File not found:", file_path)

## 12. Show Latest Predict Folder Images Inside the Notebook

In [ ]:
# Show the images from the newest predict folder.
image_exts = {".jpg", ".jpeg", ".png", ".bmp"}

predict_folders = sorted(
    [folder for folder in detect_root.glob("predict*") if folder.is_dir()],
    key=lambda folder: folder.stat().st_mtime,
    reverse=True,
)

if not predict_folders:
    print("No predict folders were found.")
else:
    latest_predict_folder = predict_folders[0]
    print("Latest predict folder:", latest_predict_folder)

    latest_predict_images = sorted(
        [file for file in latest_predict_folder.iterdir() if file.is_file() and file.suffix.lower() in image_exts]
    )

    if not latest_predict_images:
        print("No images were found in the latest predict folder.")
    else:
        for image_file in latest_predict_images:
            print(image_file.name)
            plt.figure(figsize=(9, 7))
            plt.imshow(Image.open(image_file))
            plt.title(image_file.name)
            plt.axis("off")
            plt.show()

## Notes

This model works well for basic traffic object detection, but some similar vehicle classes can still be confused. For example, **car**, **truck**, **bus**, and **ambulance** may look similar in some images, especially when the object is small, far away, blocked, or the image quality is not clear.